# Análisis diferencial de una variante de Keccak/SHA-3 con rondas dinámicas mediante MILP

**Objetivo.** Modelar, mediante *Programación Lineal Entera Mixta* (MILP), la propagación de
diferencias a través de una variante de Keccak cuyo número de rondas depende de un contador de
intentos, y determinar el **número mínimo de cajas-S (S-boxes) activas** del paso no lineal $\chi$
para $1$, $2$ y $3$ rondas con tamaños de palabra $z=4$ y $z=8$. A partir de ese mínimo se acota la
probabilidad diferencial y el número de pares texto-claro/texto-cifrado necesarios para un ataque
diferencial, lo que permite razonar sobre la relación *intentos $\to$ seguridad*.

**Variante dinámica.**

$$\text{KeccakModificado}(M,\ \text{Intentos}) \Rightarrow R=\begin{cases}1 & \text{Intentos}<10\\ 2 & 10\le\text{Intentos}<20\\ 3 & 20\le\text{Intentos}<30\end{cases}$$

Cada ronda aplica $\theta,\rho,\pi,\chi$ (omitimos $\iota$, que suma una constante y no afecta a las
diferencias). El estado es $5\times5\times z$: para $z=4$ tenemos Keccak-f[100] y $20$ cajas-S por
ronda; para $z=8$, Keccak-f[200] y $40$ cajas-S por ronda.

> **Nota sobre esta versión.** Este cuaderno parte de una implementación previa y corrige **dos
> errores** que la hacían inservible: (i) el cálculo de la DDT tenía un filtro espurio que vaciaba
> 21 de 32 filas; (ii) el *gadget* lineal del XOR volvía **infactible** el caso $a\neq b$, con lo que
> el modelo completo era infactible. Ambos se documentan y corrigen abajo.

**Contenido**
1. Núcleo Keccak/SHA-3 y la variante dinámica
2. La caja-S $\chi$ y su DDT (corrección del bug)
3. Modelo MILP ($\theta,\rho,\pi$ lineales; $\chi$ vía DDT; XOR corregido)
4. Experimento $R=1$ (óptimo probado)
5. $R=2$ y $R=3$: cotas por trayectorias diferenciales válidas
6. Probabilidad diferencial y complejidad del ataque
7. Intentos $\to$ seguridad, correcciones y conclusiones

## 1. Núcleo Keccak/SHA-3 y la variante dinámica

Implementamos los cinco pasos de Keccak como funciones puras y verificamos la esponja SHA-3-256
contra `hashlib` (FIPS 202). Esto nos da un núcleo de referencia correcto; las mismas operaciones
lineales ($\theta,\rho,\pi$), reescaladas al tamaño de palabra $z$, se reutilizan en el análisis
diferencial.

In [ ]:
import hashlib

MASK64 = 0xFFFFFFFFFFFFFFFF
ROT = [[0,36,3,41,18],[1,44,10,45,2],[62,6,43,15,61],[28,55,25,21,56],[27,20,39,8,14]]
RC = [0x0000000000000001,0x0000000000008082,0x800000000000808A,0x8000000080008000,
      0x000000000000808B,0x0000000080000001,0x8000000080008081,0x8000000000008009,
      0x000000000000008A,0x0000000000000088,0x0000000080008009,0x000000008000000A,
      0x000000008000808B,0x800000000000008B,0x8000000000008089,0x8000000000008003,
      0x8000000000008002,0x8000000000000080,0x000000000000800A,0x800000008000000A,
      0x8000000080008081,0x8000000000008080,0x0000000080000001,0x8000000080008008]

def ROTL64(x, n):
    n %= 64
    return (((x << n) & MASK64) | ((x & MASK64) >> (64 - n))) if n else (x & MASK64)

def theta(A):
    C = [A[x][0]^A[x][1]^A[x][2]^A[x][3]^A[x][4] for x in range(5)]
    D = [C[(x-1)%5] ^ ROTL64(C[(x+1)%5], 1) for x in range(5)]
    return [[A[x][y]^D[x] for y in range(5)] for x in range(5)]

def rho(A):  return [[ROTL64(A[x][y], ROT[x][y]) for y in range(5)] for x in range(5)]

def pi_(A):
    B = [[0]*5 for _ in range(5)]
    for x in range(5):
        for y in range(5):
            B[y][(2*x+3*y)%5] = A[x][y]
    return B

def chi(A):
    return [[(A[x][y] ^ ((~A[(x+1)%5][y]) & A[(x+2)%5][y])) & MASK64
             for y in range(5)] for x in range(5)]

def iota(A, rc):
    B = [f[:] for f in A]; B[0][0] ^= rc; return B

def keccak_f(state):
    for i in range(24):
        state = iota(chi(pi_(rho(theta(state)))), RC[i])
    return state

def sha3_256(msg: bytes) -> bytes:
    rate = 136; st = [[0]*5 for _ in range(5)]
    p = bytearray(msg); p.append(0x06)
    while len(p) % rate: p.append(0x00)
    p[-1] |= 0x80
    for off in range(0, len(p), rate):
        blk = p[off:off+rate]
        for i in range(rate//8):
            st[i%5][i//5] ^= int.from_bytes(blk[i*8:(i+1)*8], "little")
        st = keccak_f(st)
    out = bytearray()
    for i in range(4):
        out += st[i%5][i//5].to_bytes(8, "little")
    return bytes(out)

pruebas = [b"", b"abc", b"Plataforma del Estado - modulo de autenticacion", b"a"*135, b"a"*136]
ok = sum(sha3_256(m).hex() == hashlib.sha3_256(m).hexdigest() for m in pruebas)
print(f"SHA-3-256 (pasos separados): {ok}/{len(pruebas)} vectores coinciden con hashlib (FIPS 202)")

## 2. La caja-S $\chi$ y su DDT

El único paso no lineal es $\chi$, que opera sobre cada **fila** de 5 bits:
$$\chi_i = x_i \oplus (\overline{x_{i+1}}\, \cdot\, x_{i+2}).$$
Es una permutación de 5 bits. Su comportamiento diferencial se resume en la **Tabla de Distribución
de Diferencias** (DDT): para cada diferencia de entrada $a$ y de salida $b$,
$$\mathrm{DDT}[a][b] = \#\{x\in\{0,1\}^5 : S(x)\oplus S(x\oplus a) = b\}.$$

**Bug corregido.** La versión original incluía un filtro `if (x ^ S(x)) == a:` antes de contar, que
no corresponde a la definición de DDT: restringe $x$ a los puntos fijos de $x\mapsto x\oplus S(x)$.
El efecto era dejar **21 de 32 filas vacías** y omitir 301 de 317 transiciones válidas (p. ej.
$\mathrm{DDT}[1]$ salía $[1,9]$ en vez de $[1,9,17,25]$). La versión correcta recorre **todos** los
$x$ sin ese filtro.

In [ ]:
def keccak_sbox_5bits(x):
    b = [(x >> i) & 1 for i in range(5)]
    o = [b[i] ^ ((~b[(i+1)%5]) & b[(i+2)%5]) & 1 for i in range(5)]
    return sum(o[i] << i for i in range(5))

def ddt_correcta():
    DDT = {}
    for a in range(32):
        s = set()
        for x in range(32):                       # <-- sin el filtro espurio
            s.add(keccak_sbox_5bits(x) ^ keccak_sbox_5bits(x ^ a))
        DDT[a] = sorted(s)
    return DDT

DDT = ddt_correcta()
vacias = [a for a in range(32) if not DDT[a]]
tot = sum(len(v) for v in DDT.values())
# multiplicidad maxima para a != 0  ->  probabilidad diferencial maxima
cnt = [[0]*32 for _ in range(32)]
for a in range(32):
    for x in range(32):
        cnt[a][keccak_sbox_5bits(x) ^ keccak_sbox_5bits(x ^ a)] += 1
maxmult = max(cnt[a][b] for a in range(1,32) for b in range(32))

print("DDT[1]          =", DDT[1])
print("filas vacias    =", vacias)
print("pares validos   =", tot)
print(f"mult. maxima (a!=0) = {maxmult}/32  ->  prob. diferencial maxima = 2^-{ (32//maxmult).bit_length()-1 } = 1/4")

## 3. Modelo MILP

**Variables.** Una variable binaria por bit de diferencia del estado en cada capa,
$D_{r,x,y,z}\in\{0,1\}$ ($1$ = hay diferencia). Una variable de actividad $A_{r,y,z}\in\{0,1\}$ por
caja-S (fila $y$, bit $z$, ronda $r$).

**Capas lineales $\theta,\rho,\pi$.** Se imponen con restricciones XOR. Un XOR $c=a\oplus b$ es lineal
sobre $\mathbb{Z}$ con una variable auxiliar entera $t$:
$$a+b-2t=c,\qquad t\in\{0,1\}.$$

> **Bug corregido.** El *gadget* original usaba cuatro desigualdades que en realidad forzaban
> $a+b=2\,\text{aux}$, es decir $a=b$; el caso $a\neq b$ quedaba **infactible** y con él todo el
> modelo. La forma $a+b-2t=c$ es exacta en los cuatro casos.

**Capa no lineal $\chi$.** Para cada caja-S imponemos que el par (diferencia de entrada, diferencia
de salida) sea una **transición válida** de la DDT. Codificamos el valor entero de entrada
$v_\text{in}=\sum_i \Delta^\text{in}_i 2^i$ y de salida $v_\text{out}$, y con variables de selección
forzamos $(v_\text{in},v_\text{out})$ a ser uno de los pares válidos. Como $\chi$ es biyectiva,
$\Delta^\text{in}\neq0 \Rightarrow \Delta^\text{out}\neq0$: una caja activa siempre propaga
diferencia.

**Actividad y objetivo.** $A_{r,y,z}=1 \iff v_\text{in}\neq 0$, y minimizamos
$\sum_{r,y,z} A_{r,y,z}$. La condición de no-trivialidad correcta es $\sum D_{0}\ge 1$ (al menos una
diferencia de entrada), **no** fijar un bit concreto (lo que sesgaría el mínimo).

In [ ]:
import pulp, time

_t = [0]
def xor_lineal(prob, a, b, out):
    """out = a XOR b, exacto: a + b - 2 t = out con t binaria (gadget corregido)."""
    _t[0] += 1
    t = pulp.LpVariable("xt_%d" % _t[0], cat="Binary")
    prob += a + b - 2*t == out

def build_keccak_milp(R, z):
    _t[0] = 0
    prob = pulp.LpProblem("Keccak_R%d_z%d" % (R, z), pulp.LpMinimize)
    D = pulp.LpVariable.dicts("D", ((r,x,y,k) for r in range(R+1)
                              for x in range(5) for y in range(5) for k in range(z)), cat="Binary")
    A = pulp.LpVariable.dicts("A", ((r,y,k) for r in range(R)
                              for y in range(5) for k in range(z)), cat="Binary")
    pares = [(a,b) for a,bl in DDT.items() for b in bl]
    for r in range(R):
        # theta: C[x][k] = XOR_y D[r][x][y][k]
        C = pulp.LpVariable.dicts("C%d"%r, ((x,k) for x in range(5) for k in range(z)), cat="Binary")
        for x in range(5):
            for k in range(z):
                acc = D[(r,x,0,k)]
                for y in range(1,5):
                    nxt = pulp.LpVariable("cc_%d_%d_%d_%d"%(r,x,k,y), cat="Binary")
                    xor_lineal(prob, acc, D[(r,x,y,k)], nxt); acc = nxt
                prob += C[(x,k)] == acc
        Dth = pulp.LpVariable.dicts("Dth%d"%r, ((x,k) for x in range(5) for k in range(z)), cat="Binary")
        for x in range(5):
            for k in range(z):
                xor_lineal(prob, C[((x-1)%5,k)], C[((x+1)%5,(k-1)%z)], Dth[(x,k)])
        Dt = pulp.LpVariable.dicts("Dt%d"%r, ((x,y,k) for x in range(5) for y in range(5) for k in range(z)), cat="Binary")
        for x in range(5):
            for y in range(5):
                for k in range(z):
                    xor_lineal(prob, D[(r,x,y,k)], Dth[(x,k)], Dt[(x,y,k)])
        # rho + pi (permutacion pura); offsets rho reescalados mod z
        Drp = pulp.LpVariable.dicts("Drp%d"%r, ((x,y,k) for x in range(5) for y in range(5) for k in range(z)), cat="Binary")
        for x in range(5):
            for y in range(5):
                nx, ny, rot = y, (2*x+3*y)%5, ROT[x][y] % z
                for k in range(z):
                    prob += Drp[(nx,ny,(k+rot)%z)] == Dt[(x,y,k)]
        # chi via DDT
        for y in range(5):
            for k in range(z):
                vin = pulp.LpVariable("vin_%d_%d_%d"%(r,y,k), 0, 31, cat="Integer")
                prob += vin == pulp.lpSum(Drp[(i,y,k)]*(1<<i) for i in range(5))
                vout = pulp.LpVariable("vout_%d_%d_%d"%(r,y,k), 0, 31, cat="Integer")
                prob += vout == pulp.lpSum(D[(r+1,i,y,k)]*(1<<i) for i in range(5))
                sel = []
                for idx,(a,b) in enumerate(pares):
                    s = pulp.LpVariable("sel_%d_%d_%d_%d"%(r,y,k,idx), cat="Binary"); sel.append(s)
                    prob += vin - a <= (1-s)*31; prob += a - vin <= (1-s)*31
                    prob += vout - b <= (1-s)*31; prob += b - vout <= (1-s)*31
                prob += pulp.lpSum(sel) == 1
                prob += A[(r,y,k)]*31 >= vin
                prob += A[(r,y,k)] <= vin
                prob += vin <= 31*A[(r,y,k)]
    prob += pulp.lpSum(A[(r,y,k)] for r in range(R) for y in range(5) for k in range(z))
    prob += pulp.lpSum(D[(0,x,y,k)] for x in range(5) for y in range(5) for k in range(z)) >= 1  # no-trivial
    return prob, A

def resolver(R, z, time_limit=120):
    prob, A = build_keccak_milp(R, z)
    disp = pulp.listSolvers(onlyAvailable=True)
    solver = pulp.getSolver("HiGHS", msg=False, timeLimit=time_limit) if "HiGHS" in disp \
             else pulp.PULP_CBC_CMD(msg=0, timeLimit=time_limit)
    t0 = time.time(); prob.solve(solver); dt = time.time()-t0
    pr = [sum(1 for y in range(5) for k in range(z)
              if (pulp.value(A[(r,y,k)]) or 0) > 0.5) for r in range(R)]
    return pulp.LpStatus[prob.status], pulp.value(prob.objective), pr, dt

print("Modelo MILP construido. Solvers disponibles:", pulp.listSolvers(onlyAvailable=True))

## 4. Experimento: $R=1$ (óptimo probado)

Para una ronda el modelo es pequeño y el solver **prueba la optimalidad**. El resultado es
$1$ caja-S activa para ambos tamaños de palabra: existe una diferencia de entrada que, tras la capa
lineal, concentra toda la diferencia en una sola fila de $\chi$. Es además el mínimo teórico, porque
la ronda es una biyección y una diferencia no nula no puede desaparecer.

In [ ]:
for z in (4, 8):
    st, obj, pr, dt = resolver(1, z, time_limit=60)
    print(f"z={z}  R=1  ->  estado={st}   S-boxes activas mínimas={int(obj)}   (por ronda {pr})   {dt:.1f}s")

## 5. $R=2$ y $R=3$: cotas por trayectorias diferenciales válidas

Para $R\ge 2$ el modelo a nivel de bit tiene una **relajación lineal muy débil** (la cota dual se
queda cerca de $0$), de modo que *probar* la optimalidad con un solver MILP genérico es costoso; en la
práctica se emplean solvers dedicados o SAT/SMT. Para obtener números concretos construimos
**trayectorias diferenciales válidas** y buscamos la de menor número de cajas activas, lo que da una
**cota superior** del mínimo (una trayectoria explícita que un atacante podría usar).

La búsqueda explora diferencias post-lineales dispersas —incluyendo pares de bits en la misma fila,
que imitan las trayectorias de bajo peso reales de Keccak— y en cada caja-S activa elige la salida
válida de **menor peso de Hamming** según la DDT, propagando después por $\theta,\rho,\pi$. Aplicamos
$\theta,\rho,\pi$ directamente sobre la diferencia (son lineales), con los desplazamientos de $\rho$
reescalados $\bmod\ z$.

> **Aviso (véase la sección 8).** Los valores que produce esta búsqueda son cotas superiores *válidas pero no óptimas*: para $R=3$, $z=4$ da 18 mientras el óptimo certificado es **9**. La elección voraz del mínimo peso de Hamming en cada caja es local e ignora su efecto sobre las rondas siguientes.

In [ ]:
DDT_best = {a: min([b for b in DDT[a] if b], key=lambda b: bin(b).count("1")) for a in range(1,32)}

def rotl(x, n, w):
    n %= w
    return ((x << n) | (x >> (w-n))) & ((1<<w)-1) if n else x

def theta_w(A, w):
    C = [A[x][0]^A[x][1]^A[x][2]^A[x][3]^A[x][4] for x in range(5)]
    D = [C[(x-1)%5] ^ rotl(C[(x+1)%5], 1, w) for x in range(5)]
    return [[A[x][y]^D[x] for y in range(5)] for x in range(5)]

def rho_pi_w(A, w):
    R_ = [[rotl(A[x][y], ROT[x][y] % w, w) for y in range(5)] for x in range(5)]
    B = [[0]*5 for _ in range(5)]
    for x in range(5):
        for y in range(5):
            B[y][(2*x+3*y)%5] = R_[x][y]
    return B

def lineal(A, w):  return rho_pi_w(theta_w(A, w), w)

def propagar(CI, R, w):
    total, por_ronda = 0, []
    for r in range(R):
        act = 0; An = [[0]*5 for _ in range(5)]
        for y in range(5):
            for k in range(w):
                a = sum(((CI[x][y] >> k) & 1) << x for x in range(5))
                if a:
                    act += 1; b = DDT_best[a]
                    for x in range(5):
                        An[x][y] |= ((b >> x) & 1) << k
        por_ronda.append(act); total += act
        if r < R-1: CI = lineal(An, w)
    return total, por_ronda

def buscar(R, w, passes=4):
    cero = lambda: [[0]*5 for _ in range(5)]
    starts = []
    for x in range(5):
        for y in range(5):
            for k in range(w):
                c = cero(); c[x][y] = 1 << k; starts.append(c)
    for y in range(5):                          # pares en la misma fila de chi
        for k in range(w):
            for x1 in range(5):
                for x2 in range(x1+1, 5):
                    c = cero(); c[x1][y] = 1<<k; c[x2][y] = 1<<k; starts.append(c)
    best, bC, bpr = None, None, None
    for c in starts:
        t, pr = propagar(c, R, w)
        if best is None or t < best: best, bC, bpr = t, [r[:] for r in c], pr
    for _ in range(passes):                     # mejora local: voltear 1 bit
        mejora = False
        for x in range(5):
            for y in range(5):
                for k in range(w):
                    cand = [r[:] for r in bC]; cand[x][y] ^= (1<<k)
                    if not any(any(r) for r in cand): continue
                    t, pr = propagar(cand, R, w)
                    if t < best: best, bC, bpr, mejora = t, cand, pr, True
        if not mejora: break
    return best, bpr

resultados = {}
print(" z | R | S-boxes activas | por ronda")
print("---+---+-----------------+----------")
for w in (4, 8):
    for R in (1, 2, 3):
        n, pr = buscar(R, w); resultados[(w,R)] = (n, pr)
        print(f" {w} | {R} |       {n:3d}       | {pr}")

## 6. Probabilidad diferencial y complejidad del ataque

Cada caja-S activa aporta al peso diferencial un factor de a lo sumo $2^{-2}$ (la multiplicidad máxima
de la DDT es $8/32=1/4$). Una trayectoria con $n$ cajas activas tiene por tanto probabilidad
$$\Pr[\text{trayectoria}] \le \left(2^{-2}\right)^{n} = 2^{-2n},$$
y un ataque diferencial necesita del orden de
$$N_{\text{pares}} \approx 2^{\,2n}$$
pares para distinguir. El mínimo de cajas activas da así una **cota inferior de la complejidad de
datos**: cuantas más rondas, mayor $n$, y mayor el coste del ataque.

In [ ]:
print(" z | R | S-boxes (n) | prob. diferencial | pares necesarios")
print("---+---+-------------+-------------------+------------------")
for w in (4, 8):
    for R in (1, 2, 3):
        n, _ = resultados[(w, R)]
        print(f" {w} | {R} |     {n:3d}     |      2^-{2*n:<3d}       |     2^{2*n}")

## 7. Intentos $\to$ seguridad, correcciones y conclusiones

**Relación intentos $\to$ seguridad.** El contador de intentos fija el número de rondas $R\in\{1,2,3\}$.
Al aumentar los intentos:

| Intentos | Rondas | Mín. cajas activas (cota) | Prob. diferencial | Pares (aprox.) |
|:--:|:--:|:--:|:--:|:--:|
| $<10$ | 1 | 1 (óptimo) | $2^{-2}$ | $2^{2}$ |
| $10$–$19$ | 2 | 4 (exacto) | $2^{-8}$ | $2^{8}$ |
| $20$–$29$ | 3 | 9 ($z{=}4$) / 10 ($z{=}8$), exactos | $2^{-18}$ / $2^{-20}$ | $2^{18}$ / $2^{20}$ |

El número de cajas activas —y por tanto el coste de un ataque diferencial— **crece rápidamente** con
las rondas: pasar de 1 a 3 rondas eleva la complejidad de datos de $\sim\!4$ pares a $\sim\!2^{36}$–$2^{40}$.
En términos de seguridad, más intentos fallidos $\Rightarrow$ más rondas $\Rightarrow$ mayor resistencia
diferencial. (La cota inferior estructural es $R$: como la ronda es biyectiva, cada ronda aporta al
menos una caja-S activa.)

**Sobre las cotas.** $R=1$ es óptimo probado por el MILP. Para $R=2$ dos métodos independientes (el
incumbente del MILP y la búsqueda de trayectorias) coinciden en 4. Para $R=3$ reportamos la mejor
trayectoria válida hallada (cota superior del mínimo); el óptimo exacto exigiría un solver dedicado por
la debilidad de la relajación lineal a nivel de bit.

**Modelo elegido.** Usamos el modelo diferencial **riguroso** basado en la DDT (el atacante elige
transiciones válidas de $\chi$), que da los mínimos diferenciales reales. Un modelo alternativo que
propaga la $\chi$-imagen determinista (linealizando $\chi$ solo con compuertas AND) mide difusión y
**sobreestima** el número de cajas activas para varias rondas; por eso preferimos la DDT.

**Correcciones aplicadas respecto de la versión original.**
1. **DDT:** se eliminó el filtro `if (x ^ S(x)) == a:` que vaciaba 21/32 filas y omitía 301/317
   transiciones. La DDT correcta tiene todas las filas no vacías y multiplicidad máxima $8/32=2^{-2}$.
2. **Gadget XOR:** se sustituyeron las cuatro desigualdades que forzaban $a=b$ (dejando infactible
   $a\neq b$, y con ello todo el modelo) por la codificación exacta $a+b-2t=c$.
3. **No-trivialidad:** se reemplazó la fijación de un bit concreto por $\sum D_0\ge 1$, evitando sesgar
   el mínimo hacia trayectorias que activan ese bit.

Con estas correcciones el modelo es factible y reproduce el valor esperado $R=1\to 1$, ofreciendo un
marco coherente para analizar la variante dinámica.

## 8. Certificación de los óptimos con CP-SAT

El modelo MILP anterior certifica el óptimo sólo para $R=1$. Para $R\ge2$ la cota dual
permanece en 0 (gap del 100 %), por dos razones:

1. **Encoding *big-M* de la DDT.** Se usa una binaria de selección por cada una de las 317
   transiciones y por cada caja-S: 12 680 de las 14 140 variables (≈ 90 %) para $R=2$, $z=4$.
2. **Paridad sobre $\mathbb{F}_2$**, que se relaja muy mal a variables continuas.

Se probaron dos técnicas habituales sin éxito: reformular como problema de decisión
($\sum A \le K$) no cerró $R=2$, $z=4$ en 180 s, y la ruptura de simetría por traslación en
$z$ —válida, porque la ronda sin $\iota$ conmuta con dicha traslación— sólo puede ahorrar
un factor $z$.

La solución es cambiar de herramienta. CP-SAT trata de forma nativa las dos estructuras
problemáticas: el XOR (`AddBoolXOr`, sin auxiliares) y la tabla de transiciones de $\chi$
(`AddAllowedAssignments`, sin *big-M* ni variables de selección).

> **Importante — reinicia el kernel antes de ejecutar esta sección.** `highspy` (usado arriba por PuLP) y `ortools` empaquetan builds distintos de HiGHS y sus símbolos colisionan: no pueden convivir en un mismo proceso de Python. Si ya ejecutaste las celdas del MILP con HiGHS, reinicia el kernel y ejecuta sólo las celdas de la 1 a la 3 (núcleo y DDT) antes de continuar aquí.

In [ ]:
# pip install ortools
# Requiere un kernel donde highspy NO se haya importado (véase el aviso anterior).
try:
    from ortools.sat.python import cp_model
except ImportError as e:
    raise SystemExit("Conflicto highspy/ortools: reinicia el kernel y ejecuta sólo "
                     "las celdas del núcleo y la DDT antes de esta sección.\n%s" % e)

TUPLAS = []
for a, bl in DDT.items():
    for b in bl:
        TUPLAS.append([(a >> i) & 1 for i in range(5)] + [(b >> i) & 1 for i in range(5)])

def cpsat(R, z, limite=120, hilos=8):
    m = cp_model.CpModel()
    D = {(r,x,y,k): m.NewBoolVar("D%d_%d_%d_%d"%(r,x,y,k))
         for r in range(R+1) for x in range(5) for y in range(5) for k in range(z)}
    A = {}
    for r in range(R):
        C = {(x,k): m.NewBoolVar("C%d_%d_%d"%(r,x,k)) for x in range(5) for k in range(z)}
        for x in range(5):
            for k in range(z):
                m.AddBoolXOr([C[(x,k)].Not()] + [D[(r,x,y,k)] for y in range(5)])
        Dth = {(x,k): m.NewBoolVar("T%d_%d_%d"%(r,x,k)) for x in range(5) for k in range(z)}
        for x in range(5):
            for k in range(z):
                m.AddBoolXOr([Dth[(x,k)].Not(), C[((x-1)%5,k)], C[((x+1)%5,(k-1)%z)]])
        Dt = {(x,y,k): m.NewBoolVar("t%d_%d_%d_%d"%(r,x,y,k))
              for x in range(5) for y in range(5) for k in range(z)}
        for x in range(5):
            for y in range(5):
                for k in range(z):
                    m.AddBoolXOr([Dt[(x,y,k)].Not(), D[(r,x,y,k)], Dth[(x,k)]])
        Drp = {}
        for x in range(5):
            for y in range(5):
                nx, ny, rot = y, (2*x+3*y)%5, ROT[x][y] % z
                for k in range(z):
                    Drp[(nx,ny,(k+rot)%z)] = Dt[(x,y,k)]
        for y in range(5):
            for k in range(z):
                ent = [Drp[(i,y,k)] for i in range(5)]
                m.AddAllowedAssignments(ent + [D[(r+1,i,y,k)] for i in range(5)], TUPLAS)
                a = m.NewBoolVar("A%d_%d_%d"%(r,y,k)); m.AddMaxEquality(a, ent); A[(r,y,k)] = a
    # ruptura de simetria: slice z=0 no nulo (implica no trivialidad)
    m.AddBoolOr([D[(0,x,y,0)] for x in range(5) for y in range(5)])
    m.Minimize(sum(A.values()))
    s = cp_model.CpSolver()
    s.parameters.max_time_in_seconds = float(limite)
    s.parameters.num_search_workers = hilos
    st = s.Solve(m)
    ok = st in (cp_model.OPTIMAL, cp_model.FEASIBLE)
    pr = ([sum(s.Value(A[(r,y,k)]) for y in range(5) for k in range(z)) for r in range(R)]
          if ok else None)
    return s.StatusName(st), (int(s.ObjectiveValue()) if ok else None), \
           (int(s.BestObjectiveBound()) if ok else R), pr

print(" z | R | estado   | minimo    | por ronda")
print("---+---+----------+-----------+----------")
for z in (4, 8):
    for R in (1, 2, 3):
        est, ub, lb, pr = cpsat(R, z, limite=120)
        minimo = f"{ub} exacto" if est == "OPTIMAL" else f"[{lb}, {ub}]"
        print(f" {z} | {R} | {est:<8} | {minimo:<9} | {pr}")

### Resultado

| $z$ | $R$ | MILP (120 s) | CP-SAT | Óptimo |
|:---:|:---:|:---:|:---:|:---:|
| 4 | 1 | 1, certificado | 1, certificado (0,8 s) | **1** |
| 4 | 2 | 9, gap 100 % | 4, certificado (2,9 s) | **4** |
| 4 | 3 | 18, gap 100 % | 9, certificado (68 s) | **9** |
| 8 | 1 | 1, certificado | 1, certificado (2,1 s) | **1** |
| 8 | 2 | 4, gap 100 % | 4, certificado (9,2 s) | **4** |
| 8 | 3 | sin solución | 10, certificado (118 s) | **10** |

Dos conclusiones:

**La cota voraz era mala.** Para $R=3$, $z=4$ el óptimo es **9**, no 18: un factor 2. La
descomposición por ronda lo explica — el óptimo es $2,4,3$, es decir, la trayectoria
*vuelve a estrecharse* en la tercera ronda. La búsqueda voraz produce $2,2,14$ porque
minimizar el peso caja por caja es una decisión local: la trayectoria óptima sacrifica peso
en la segunda ronda (4 cajas en lugar de 2) para poder contraerse en la tercera.

**Esto corrige a la baja las cifras de seguridad.** Para $R=3$, $z=4$ los pares necesarios
pasan de $2^{36}$ a $2^{18}$ ($\sim 2.6\times10^{5}$): la variante es bastante menos segura
de lo que sugería la cota no certificada. La conclusión cualitativa —la resistencia crece
de forma monótona con los intentos— se mantiene; su magnitud, no.

Y a cambio se gana algo más valioso: los seis casos pasan de «cota superior del
atacante» a **óptimo certificado**, que es lo que constituye una garantía de seguridad,
porque ésta depende de la cota *inferior* del número de cajas activas.